# HouseCrafter Colab — 2D floorplan to 3D `.ply`

Run **every cell in order** (Runtime → GPU). Then open the Gradio link, paste a Pinterest **image** URL, click Generate.

On Pinterest: open the pin → right-click the floorplan picture → **Copy image address** (`i.pinimg.com/...`). A pin page URL (`pinterest.com/pin/...`) also works if it has `og:image`.

Do **not** `pip install -r requirements.txt` and do **not** compile PyTorch3D.

## Step 1 — GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

## Step 2 — Google Drive (outputs only)
Creates `MyDrive/Gradio/houseCrafter/output`. Checkpoints are **not** cached on Drive.

In [ ]:
from pathlib import Path

GDRIVE_OUT = Path("/content/drive/MyDrive/Gradio/houseCrafter/output")
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("OK Drive output:", GDRIVE_OUT)
except Exception as exc:
    GDRIVE_OUT = Path("/content/houseCrafter_output")
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Drive unavailable, local:", GDRIVE_OUT, exc)

## Step 3 — Get the code

In [ ]:
from pathlib import Path

REPO = Path("/content/houseCrafter")
BRANCH = "feat/gradio-colab-ui"
URL = "https://github.com/sourman-dev/houseCrafter.git"

if not (REPO / "app.py").exists():
    %cd /content
    !git clone --branch {BRANCH} --single-branch {URL} houseCrafter || git clone {URL} houseCrafter

%cd /content/houseCrafter
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git log -1 --oneline
assert Path("app.py").exists()

## Step 4 — Install (wheels only)
`open3d` will SKIP on Python 3.13. That is OK.

In [ ]:
%cd /content/houseCrafter
!bash scripts/colab_setup.sh
import gradio
print("gradio", gradio.__version__)

## Step 5 — Launch Gradio
Wait for a `*.gradio.live` link. Then:
1. Choose **Paste image URL**
2. Paste the Pinterest **image address**
3. Click **Load image from URL** (preview)
4. Click **Generate 3D Scene**
5. Orbit the `.ply` and download it. Copy is also saved under Drive `Gradio/houseCrafter/output`.

In [ ]:
%cd /content/houseCrafter
import os
from pathlib import Path

out = "/content/drive/MyDrive/Gradio/houseCrafter/output"
if not Path(out).exists():
    out = "/content/houseCrafter_output"
    Path(out).mkdir(parents=True, exist_ok=True)

os.environ["GDRIVE_OUTPUT_DIR"] = out
!python app.py --share --server_name 0.0.0.0 --gdrive_dir "{out}"

## Step 6 — List saved `.ply` files

In [ ]:
from pathlib import Path

out = Path("/content/drive/MyDrive/Gradio/houseCrafter/output")
if not out.exists():
    out = Path("/content/houseCrafter_output")
print("listing", out)
if out.exists():
    for p in sorted(out.rglob("*.ply"))[-12:]:
        print(p, p.stat().st_size)